# 11.2 Convolutional neural networks
# 11.2.1 2D cross-correlation operation

In [ ]:
import torch

def corr2d(X, K):
    """Compute 2D cross-correlation."""
    h, w = K.shape # Kernel height and width
    n_h, n_w = X.shape # Input height and width
    out_h = n_h - h + 1
    out_w = n_w - w + 1
    Y = torch.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            # Extract the relevant sub-tensor from input X
            X_sub = X[i:i + h, j:j + w]
            # Perform element-wise multiplication and sum
            Y[i, j] = (X_sub * K).sum()
    return Y

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
output = corr2d(X, K)
print(output)

Input X:
tensor([[0., 1., 2.],
        [3., 4., 5.],
        [6., 7., 8.]])
Kernel K:
tensor([[0., 1.],
        [2., 3.]])
Output Y:
tensor([[19., 25.],
        [37., 43.]])


## 11.2.2 Multiple Input and Multiple Output Channels

In [3]:
def corr2d_multi_in(X, K):
    return sum(corr2d(x, k) for x, k in zip(X, K))
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], dim=0)

X_multi = torch.rand(2, 5, 5) # Input channels=2, height=5, width=5
K_multi_in = torch.rand(2, 3, 3)
K_multi_in_out = torch.rand(3, 2, 3, 3) # Output channels=3, Input channels=2, kH=3, kW=3
output_multi = corr2d_multi_in_out(X_multi, K_multi_in_out)
# Output is a 3D tensor (3 output channels)
print(f"Multi-input shape: {X_multi.shape}")
print(f"Kernel shape: {K_multi_in_out.shape}")
print(f"Output shape: {output_multi.shape}")

Multi-input shape: torch.Size([2, 5, 5])
Kernel shape: torch.Size([3, 2, 3, 3])
Output shape: torch.Size([3, 3, 3])


## 11.2.3 1x1 Convolutional Layer

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    # Reshape input and kernel for matrix multiplication
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    # Perform matrix multiplication (fully connected operation)
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

X_1x1 = torch.rand(3, 3, 3) # c_i=3, h=3, w=3
K_1x1 = torch.rand(2, 3, 1, 1) # c_o=2, c_i=3, kH=1, kW=1
output_1x1 = corr2d_multi_in_out_1x1(X_1x1, K_1x1)
print(f"1x1 Input shape: {X_1x1.shape}")
print(f"1x1 Kernel shape: {K_1x1.shape}")
print(f"1x1 Output shape: {output_1x1.shape}")

## 11.2.4 Transposed convolutional layers

In [ ]:
import torch
import torch.nn as nn
# Input tensor (batch_size, in_channels, height, width)
input_tensor = torch.randn(1, 16, 8, 8) # Input size 8x8
# Define transposed convolution layer to double the size
# Formula check: o_h = (8 - 1)*2 - 2*1 + 3 + 1 = 14 - 2 + 3 + 1 = 16
upsample_layer = nn.ConvTranspose2d(in_channels=16, 
                                    out_channels=32,
                                    kernel_size=3, stride=2,
                                    padding=1, 
                                    output_padding=1, 
                                    bias=False)

output_tensor = upsample_layer(input_tensor)
print(f"Input shape:  {input_tensor.shape}")
print(f"Output shape: {output_tensor.shape}")

Input shape:  torch.Size([1, 16, 8, 8])
Output shape: torch.Size([1, 32, 16, 16])


## 11.2.5 Padding and stride in nn.Conv2d

In [ ]:
import torch
import torch.nn as nn
# Input tensor (batch_size, in_channels, height, width)
input_tensor = torch.randn(1, 3, 32, 32) # Input size 32x32
conv_same = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
output_same = conv_same(input_tensor)
# Formula: floor((32 + 2*1 - 3)/1) + 1 = floor(31) + 1 = 32
conv_downsample = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=0)
output_downsample = conv_downsample(input_tensor)
# Formula: floor((32 + 2*0 - 3)/2) + 1 = floor(29/2) + 1 = 14 + 1 = 15
conv_downsample_pad = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1)
output_downsample_pad = conv_downsample_pad(input_tensor)
# Formula: floor((32 + 2*1 - 3)/2) + 1 = floor(31/2) + 1 = 15 + 1 = 16
print(f"Input shape: {input_tensor.shape}")
print(f"Output shape (stride=1, padding=1): {output_same.shape}")
print(f"Output shape (stride=2, padding=0): {output_downsample.shape}")
print(f"Output shape (stride=2, padding=1): {output_downsample_pad.shape}")

Input shape: torch.Size([1, 3, 32, 32])
Output shape (stride=1, padding=1): torch.Size([1, 16, 32, 32])
Output shape (stride=2, padding=0): torch.Size([1, 16, 15, 15])
Output shape (stride=2, padding=1): torch.Size([1, 16, 16, 16])


## 11.2.6 Pooling layer

In [ ]:
import torch
import torch.nn as nn
# Simple 4x4 input tensor (batch_size=1, channels=1)
input_tensor = torch.tensor([[[[ 0.,  1.,  2.,  3.],
                               [ 4.,  5.,  6.,  7.],
                               [ 8.,  9., 10., 11.],
                               [12., 13., 14., 15.]]]], dtype=torch.float32)
# Max Pooling with 2x2 window, stride 2
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
max_output = max_pool(input_tensor)
# Average Pooling with 2x2 window, stride 2
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
avg_output = avg_pool(input_tensor)

print(f"Input Tensor:\n{input_tensor}\n")
print(f"Max Pooling Output:\n{max_output}\n")
print(f"Average Pooling Output:\n{avg_output}")
print(f"\nOutput shape (both): {max_output.shape}")

## 11.2.7 Batch normalization

In [ ]:
import torch
import torch.nn as nn

# --- BatchNorm2d Example (for Convolutional Layers) ---
# Input: (batch_size, num_channels, height, width)
input_conv = torch.randn(4, 16, 20, 20)
bn2d = nn.BatchNorm2d(num_features=16)
output_conv = bn2d(input_conv)
print(f"Input shape:  {input_conv.shape}")
print(f"Output shape: {output_conv.shape}")

# --- BatchNorm1d Example (for Fully Connected Layers) ---
# Input: (batch_size, num_features)
input_fc = torch.randn(4, 100)
bn1d = nn.BatchNorm1d(num_features=100)
output_fc = bn1d(input_fc)
print(f"\n--- BatchNorm1d ---")
print(f"Input shape:  {input_fc.shape}")
print(f"Output shape: {output_fc.shape}")

Input shape:  torch.Size([4, 16, 20, 20])
Output shape: torch.Size([4, 16, 20, 20])

--- BatchNorm1d ---
Input shape:  torch.Size([4, 100])
Output shape: torch.Size([4, 100])


## 11.2.8 Construct a ResNet-18

In [ ]:
# Define the Residual block
class Residual(nn.Module):
    """The Residual block of ResNet models."""
    def __init__(self, in_channels, num_channels, 
                 use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, num_channels, 
                               kernel_size=3, padding=1, 
                               stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, 
                               kernel_size=3, padding=1)
        # Shortcut path 1x1 convolution (if needed for dimension matching)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, num_channels, 
                                   kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        # Apply main path transformations
        Y = torch.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        # Shortcut connection
        Y += X
        # Apply final ReLU activation
        return torch.relu(Y)

# Creating a ResNet stage (a stack of residual blocks)
def resnet_block(in_channels, num_channels, 
                 num_residuals, first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(in_channels, num_channels, 
                                use_1x1conv=True, strides=2))
        else:
            blk.append(Residual(num_channels, num_channels))
        in_channels = num_channels
    return nn.Sequential(*blk)

# Define ResNet-18 Architecture using the blocks
resnet18 = nn.Sequential(
    nn.Conv2d(1, 64, kernel_size=7, stride=2, 
              padding=3),
    nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, 
                 padding=1),
    # first_block=True means no downsampling here
    resnet_block(64, 64, 2, 
                 first_block=True),
    resnet_block(64, 128, 2),
    resnet_block(128, 256, 2),
    resnet_block(256, 512, 2),
    # Global Average Pooling and final Linear layer
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(512, 10)
)

# Example usage (requires appropriate input size and resources)
dummy_input_resnet = torch.randn(1, 1, 224, 224)
output_resnet = resnet18(dummy_input_resnet)
print("ResNet-18 Output Shape:", output_resnet.shape)

ResNet-18 Output Shape: torch.Size([1, 10])


## 11.2.9 Construct a DenseNet

In [9]:
def conv_block(in_channels, num_channels):
    return nn.Sequential(
        nn.BatchNorm2d(in_channels), nn.ReLU(),
        nn.Conv2d(in_channels, num_channels, kernel_size=3, padding=1)
    )

# Define the Dense Block
class DenseBlock(nn.Module):
    def __init__(self, num_convs, in_channels, num_channels):
        super(DenseBlock, self).__init__()
        layer = []
        # `num_channels` here is the 'growth_rate'
        for i in range(num_convs):
            current_in_channels = in_channels + i * num_channels
            layer.append(conv_block(current_in_channels, num_channels))
        self.net = nn.ModuleList(layer)

    def forward(self, X):
        for blk in self.net:
            Y = blk(X)
            X = torch.cat((X, Y), dim=1)
        return X

# Define the Transition Layer (reduces channels and downsamples)
def transition_block(in_channels, num_channels):
    return nn.Sequential(
        nn.BatchNorm2d(in_channels), nn.ReLU(),
        nn.Conv2d(in_channels, num_channels, kernel_size=1),
        nn.AvgPool2d(kernel_size=2, stride=2)            
    )

num_channels = 64 # Initial number of channels after stem
growth_rate = 32 # Number of channels added by each conv_block in a DenseBlock
num_convs_in_dense_blocks = [6, 12, 24, 16] # Number of convolutional layers in each dense block (e.g., for DenseNet-121)

densenet_layers = [
    # Stem: Initial Conv, BN, ReLU, MaxPool
    nn.Conv2d(1, num_channels, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(num_channels), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
]

# Add Dense Blocks and Transition Layers iteratively
for i, num_convs in enumerate(num_convs_in_dense_blocks):
    densenet_layers.append(DenseBlock(num_convs, num_channels, growth_rate))
    num_channels += num_convs * growth_rate
    if i != len(num_convs_in_dense_blocks) - 1:
        output_channels_transition = num_channels // 2
        densenet_layers.append(transition_block(num_channels, output_channels_transition))
        num_channels = output_channels_transition

densenet_layers.extend([
    nn.BatchNorm2d(num_channels), nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(num_channels, 10)
])

densenet = nn.Sequential(*densenet_layers)
dummy_input_densenet = torch.randn(1, 1, 224, 224)
output_densenet = densenet(dummy_input_densenet)
print("DenseNet Output Shape:", output_densenet.shape)

DenseNet Output Shape: torch.Size([1, 10])


## 11.2.10 Implement Fully Convolutional Network based on ResNet-18

In [14]:
import torch
import torch.nn as nn
import torchvision

# Load a pre-trained ResNet-18 model
pretrained_net = torchvision.models.resnet18(pretrained=True)

# Create the FCN by removing the final avgpool and fc layers from ResNet-18
fcn_backbone = nn.Sequential(*list(pretrained_net.children())[:-2])

num_classes = 21
# Add the final layers typical for FCN
# 1x1 convolution to map features to class scores
final_conv = nn.Conv2d(512, num_classes, kernel_size=1) # ResNet-18's layer4 outputs 512 channels

# Transposed convolution to upsample the prediction map
# Stride=32, kernel_size=64, padding=16 are common choices to upsample by 32x
# (matching the total downsampling factor of ResNet-18)
transposed_conv = nn.ConvTranspose2d(num_classes, num_classes,
                                     kernel_size=64, padding=16, stride=32)

# Combine into a simple FCN model
fcn_model = nn.Sequential(
    fcn_backbone,
    final_conv,
    transposed_conv
)

# --- Example Usage ---
# Create a dummy input image (batch_size=1, channels=3, height=320, width=480)
dummy_input_fcn = torch.randn(1, 3, 320, 480)
output_fcn = fcn_model(dummy_input_fcn)
# The output shape should match the input spatial dimensions, with channels = num_classes
print("FCN Input Shape:", dummy_input_fcn.shape)
print("FCN Output Shape:", output_fcn.shape)
# Expected: FCN Output Shape: torch.Size([1, 21, 320, 480]) or similar depending on exact layers
# Note: The exact output size might need slight adjustments depending on padding/kernel choices
#       and how the backbone handles dimensions. Bilinear upsampling is also common.

FCN Input Shape: torch.Size([1, 3, 320, 480])
FCN Output Shape: torch.Size([1, 21, 320, 480])


## 11.2.11 Construct a U-Net

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define a double convolution block used repeatedly in U-Net
def double_conv(in_channels, out_channels):
    # Using padding=1 allows concatenation without cropping if input size is compatible
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

# Define the U-Net architecture
class UNet(nn.Module):
    def __init__(self, n_channels, n_classes):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        # --- Contracting Path (Encoder) ---
        self.inc = double_conv(n_channels, 64)      # Initial convolution block
        self.down1 = nn.Sequential(nn.MaxPool2d(2), double_conv(64, 128))  # Downsample + conv block
        self.down2 = nn.Sequential(nn.MaxPool2d(2), double_conv(128, 256)) # Downsample + conv block
        self.down3 = nn.Sequential(nn.MaxPool2d(2), double_conv(256, 512)) # Downsample + conv block
        # Bottleneck (can be another downsample or just convolutions)
        self.down4 = nn.Sequential(nn.MaxPool2d(2), double_conv(512, 1024)) # Example uses 1024 channels

        # --- Expansive Path (Decoder) ---
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2) # Upsample
        # Double conv block taking concatenated input (upsampled + skip connection)
        self.conv1 = double_conv(1024, 512) # Input channels = 512 (from up1) + 512 (from down3)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)  # Upsample
        self.conv2 = double_conv(512, 256)  # Input channels = 256 (from up2) + 256 (from down2)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)  # Upsample
        self.conv3 = double_conv(256, 128)  # Input channels = 128 (from up3) + 128 (from down1)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)   # Upsample
        self.conv4 = double_conv(128, 64)   # Input channels = 64 (from up4) + 64 (from inc)

        # --- Final Layer ---
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1) # 1x1 convolution to map to output classes

    def forward(self, x):
        # --- Encoder ---
        x1 = self.inc(x)    # Output shape: (B, 64, H, W)
        x2 = self.down1(x1) # Output shape: (B, 128, H/2, W/2)
        x3 = self.down2(x2) # Output shape: (B, 256, H/4, W/4)
        x4 = self.down3(x3) # Output shape: (B, 512, H/8, W/8)
        x5 = self.down4(x4) # Output shape: (B, 1024, H/16, W/16) - Bottleneck

        # --- Decoder + Skip Connections ---
        # Upsample x5, concatenate with x4, then apply double_conv
        up_x1 = self.up1(x5)            # Output shape: (B, 512, H/8, W/8)
        # Ensure spatial dimensions match for concatenation (may need padding/cropping if not using padding in convs)
        concat1 = torch.cat([up_x1, x4], dim=1) # Output shape: (B, 1024, H/8, W/8)
        dec_x1 = self.conv1(concat1)     # Output shape: (B, 512, H/8, W/8)

        up_x2 = self.up2(dec_x1)         # Output shape: (B, 256, H/4, W/4)
        concat2 = torch.cat([up_x2, x3], dim=1) # Output shape: (B, 512, H/4, W/4)
        dec_x2 = self.conv2(concat2)     # Output shape: (B, 256, H/4, W/4)

        up_x3 = self.up3(dec_x2)         # Output shape: (B, 128, H/2, W/2)
        concat3 = torch.cat([up_x3, x2], dim=1) # Output shape: (B, 256, H/2, W/2)
        dec_x3 = self.conv3(concat3)     # Output shape: (B, 128, H/2, W/2)

        up_x4 = self.up4(dec_x3)         # Output shape: (B, 64, H, W)
        concat4 = torch.cat([up_x4, x1], dim=1) # Output shape: (B, 128, H, W)
        dec_x4 = self.conv4(concat4)     # Output shape: (B, 64, H, W)

        # Apply final 1x1 convolution
        logits = self.outc(dec_x4)       # Output shape: (B, n_classes, H, W)
        return logits

unet_model = UNet(n_channels=3, n_classes=2)
dummy_input_unet = torch.randn(1, 3, 256, 256)

# Pass through the U-Net model
output_unet = unet_model(dummy_input_unet)

# Output shape should match input spatial dimensions, with channels = n_classes
print("U-Net Input Shape:", dummy_input_unet.shape)
print("U-Net Output Shape:", output_unet.shape) # Expected: torch.Size([1, 2, 256, 256])

U-Net Input Shape: torch.Size([1, 3, 256, 256])
U-Net Output Shape: torch.Size([1, 2, 256, 256])


## 11.2.12 One-stage detector for object detection

In [1]:
import torch
import torch.nn as nn

# A very simple CNN backbone
backbone = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2), # Downsample H, W
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2), # Downsample H, W again
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU()
)

num_classes = 5
num_anchors_per_loc = 3 # Assume 3 anchor boxes per spatial location
feature_channels = 64

# Prediction Heads (Simple Conv Layers)
# Class prediction head: predicts scores for (num_classes + 1) categories for each anchor
cls_head = nn.Conv2d(feature_channels, 
                     num_anchors_per_loc * (num_classes + 1),
                     kernel_size=3, padding=1)
# Bbox prediction head: predicts 4 offset values for each anchor
bbox_head = nn.Conv2d(feature_channels, 
                      num_anchors_per_loc * 4,
                      kernel_size=3, padding=1)

dummy_input = torch.randn(4, 3, 128, 128) # Example batch of 4 images, 128x128 pixels
features = backbone(dummy_input)
# Pass features through the prediction heads
class_predictions = cls_head(features)
bbox_offset_predictions = bbox_head(features)
print("Input Image Shape:", dummy_input.shape)
print("Backbone Feature Map Shape:", features.shape)
# Class prediction shape: (batch, num_anchors*(num_classes+1), H', W')
print("Class Head Output Shape:", class_predictions.shape)
# Bbox prediction shape: (batch, num_anchors*4, H', W')
print("Bbox Head Output Shape:", bbox_offset_predictions.shape)

Input Image Shape: torch.Size([4, 3, 128, 128])
Backbone Feature Map Shape: torch.Size([4, 64, 32, 32])
Class Head Output Shape: torch.Size([4, 18, 32, 32])
Bbox Head Output Shape: torch.Size([4, 12, 32, 32])
